# Notebook de Test - Pipeline Consommation Électrique

Test pas-à-pas des tâches du pipeline de consommation.
Utilise les configurations par environnement (ENV=dev|test|prod).

## 1. Initialisation et Configuration

In [ ]:
import os
import logging
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# Charger les variables d'environnement
load_dotenv(find_dotenv(".env"), override=True)
load_dotenv(find_dotenv(".env.secrets"), override=True)

# Configurer le logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
from ml.config import load_config

# Définir l'environnement (dev par défaut)
env = os.getenv('ENV', 'dev')
print(f'Environment: {env}')

# Charger la configuration
CONFIG_PATH = "../src/configs/consumption.yaml"
config = load_config(CONFIG_PATH)

# Afficher la configuration chargée
print('=== CONFIGURATION ===')
print(f"Raw file: {config.get('data', {}).get('raw_path')}")
print(f"Processed path: {config.get('data', {}).get('processed_path')}")
print(f"Target column: {config.get('data', {}).get('target_column')}")
print(f"Feature columns: {config.get('data', {}).get('feature_columns')}")
print(f"Model type: {config.get('model', {}).get('model_type')}")

## 2. Test du ConsumptionDataPreparer

In [ ]:
from ml.consumption.consumption_preparer import ConsumptionDataPreparer

# Initialiser le preparer (utilise la config par défaut)
preparer = ConsumptionDataPreparer()
print('✅ ConsumptionDataPreparer initialisé')

### 2.1. Chargement des données consommation (avec raw_path depuis config)

In [ ]:
# Tester le chargement avec raw_path depuis la config
try:
    # Si le fichier existe, essayer de le charger
    raw_file = config.get('data', {}).get('raw_path')
    print(f'Chargement de: {raw_file}')
    consumption_df = preparer.load_raw_consumption()  # Pas de paramètre = utilise config
    print(f'✅ Données consommation chargées: {len(consumption_df)} enregistrements')
    print(f'Colonnes: {list(consumption_df.columns)}')
    display(consumption_df.head())
except FileNotFoundError as e:
    print(f'⚠️ Fichier non trouvé: {e}')
    print('→ Utilisez un fichier existant ou modifiez la config')
except Exception as e:
    print(f'❌ Erreur: {e}')

### 2.2. Chargement avec chemin personnalisé

In [ ]:
# Tester avec un fichier de test connu
test_file = '../data/test/test_raw_consumption.csv'
if Path(test_file).exists():
    consumption_df = preparer.load_raw_consumption(raw_path=test_file)
    print(f'✅ Données consommation chargées depuis {test_file}: {len(consumption_df)} enregistrements')
    display(consumption_df.head())
else:
    print(f'⚠️ Fichier {test_file} non trouvé')

## 3. Test avec données complètes (si disponibles)

In [ ]:
# Chemins des fichiers depuis la configuration
WEATHER_FILE = config.get('data', {}).get('weather_file', '../data/processed/weather.parquet')
HOLIDAYS_FILE = config.get('data', {}).get('holidays_file', '../data/processed/holidays.parquet')
TRAIN_FILE = config.get('data', {}).get('train_path', '../data/processed/consumption/train.parquet')

# Vérifier que les fichiers existent
files_exist = {
    'weather': Path(WEATHER_FILE).exists(),
    'holidays': Path(HOLIDAYS_FILE).exists()
}
print(f'Fichiers disponibles: {files_exist}')

In [ ]:
# Si les fichiers existent, tester le pipeline complet
if all(files_exist.values()):
    print('\n>>> Test du pipeline complet (tout depuis config)...')
    try:
        # Tous les chemins viennent de la config
        features_df = preparer.prepare()  # Aucun paramètre = tout depuis config
        print(f'✅ Features générées: {len(features_df)} enregistrements')
        print(f'Colonnes: {list(features_df.columns)}')
        display(features_df.head())
        print(f'Fichier sauvegardé: {TRAIN_FILE}')
    except Exception as e:
        print(f'❌ Erreur dans le pipeline: {e}')
else:
    print('⚠️ Tous les fichiers ne sont pas disponibles pour le test complet')

## 4. Test avec différent environnement

In [ ]:
# Changer d'environnement et recharger la config
os.environ['ENV'] = 'prod'  # ou 'test'
config_prod = load_config(CONFIG_PATH)

print('=== CONFIG PROD ===')
print(f"Raw file: {config_prod.get('data', {}).get('raw_path')}")
print(f"Processed path: {config_prod.get('data', {}).get('processed_path')}")
print(f"Model name: {config_prod.get('mlflow', {}).get('model_name')}")

# Réinitialiser à dev
os.environ['ENV'] = 'dev'

## 5. Résumé

In [ ]:
print('\n=== RÉSUMÉ ===')
print('✅ Configuration chargée avec succès')
print('✅ ConsumptionDataPreparer initialisé')
print('✅ Support des environnements (dev/test/prod) fonctionnel')
print('\nProchaines étapes:')
print('- Exécuter avec ENV=dev pour le développement')
print('- Exécuter avec ENV=test pour les tests')
print('- Exécuter avec ENV=prod pour la production')